# Detect incorrectly answered questions

Choose a prediction CSV below. This notebook compares its `answer` values with the dataset's `correct_answer` values and lists every incorrect `question_id` in dataset order.

In [13]:
from __future__ import annotations

import csv
import json
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_PATH = PROJECT_ROOT / "data" / "cancermyth_screening_dataset.json"

# Change this filename to inspect another RAG result file.
ANSWERS_PATH = PROJECT_ROOT / "rag" / "rag_run_all" / "answers_oncology_expert.csv"

print(f"Dataset: {DATASET_PATH}")
print(f"Answers: {ANSWERS_PATH}")

Dataset: D:\Personal Project\RAG In Cancer Myth\rag-cancer-myth\data\cancermyth_screening_dataset.json
Answers: D:\Personal Project\RAG In Cancer Myth\rag-cancer-myth\rag\rag_run_all\answers_oncology_expert.csv


In [14]:
def parse_boolean(value: str, *, question_id: str) -> bool:
    normalized = value.strip().lower()
    if normalized == "true":
        return True
    if normalized == "false":
        return False
    raise ValueError(f"Answer for question_id={question_id} must be true or false; got {value!r}.")


with DATASET_PATH.open(encoding="utf-8") as dataset_file:
    dataset = json.load(dataset_file)

if not isinstance(dataset, list) or not dataset:
    raise ValueError("The dataset must be a non-empty JSON array.")

dataset_by_id: dict[str, dict] = {}
for record in dataset:
    question_id = str(record.get("id"))
    if question_id in dataset_by_id:
        raise ValueError(f"Duplicate question ID in dataset: {question_id}")
    if not isinstance(record.get("correct_answer"), bool):
        raise ValueError(f"correct_answer for question_id={question_id} must be Boolean.")
    dataset_by_id[question_id] = record

if not ANSWERS_PATH.is_file():
    raise FileNotFoundError(f"Answer file does not exist: {ANSWERS_PATH}")

predictions: dict[str, bool] = {}
with ANSWERS_PATH.open(newline="", encoding="utf-8") as answers_file:
    reader = csv.DictReader(answers_file)
    if reader.fieldnames != ["question_id", "answer"]:
        raise ValueError("The answer CSV must contain exactly: question_id, answer")
    for row in reader:
        question_id = row["question_id"].strip()
        if question_id not in dataset_by_id:
            raise ValueError(f"Unknown question_id in answer CSV: {question_id}")
        if question_id in predictions:
            raise ValueError(f"Duplicate question_id in answer CSV: {question_id}")
        predictions[question_id] = parse_boolean(row["answer"], question_id=question_id)

print(f"Loaded {len(predictions):,} predictions from {ANSWERS_PATH.name}.")

Loaded 735 predictions from answers_oncology_expert.csv.


In [15]:
wrong_question_ids = [
    record["id"]
    for record in dataset
    if str(record["id"]) in predictions
    and predictions[str(record["id"])] != record["correct_answer"]
]

print(f"Wrong answers: {len(wrong_question_ids):,} / {len(predictions):,}")
print("Wrong question IDs:")
print(wrong_question_ids)

Wrong answers: 223 / 735
Wrong question IDs:
[7, 11, 14, 19, 24, 27, 49, 51, 60, 67, 68, 72, 77, 78, 79, 83, 84, 85, 88, 89, 90, 91, 92, 94, 98, 102, 105, 112, 116, 119, 121, 130, 132, 142, 143, 148, 150, 153, 154, 155, 158, 162, 165, 166, 168, 177, 178, 185, 191, 192, 193, 195, 201, 205, 209, 216, 221, 225, 236, 247, 257, 265, 269, 272, 276, 277, 279, 283, 284, 285, 290, 304, 321, 327, 336, 367, 373, 376, 379, 395, 396, 415, 417, 425, 435, 439, 441, 444, 446, 448, 450, 452, 454, 462, 464, 467, 470, 472, 473, 481, 488, 493, 497, 500, 501, 505, 527, 534, 535, 536, 539, 554, 559, 572, 575, 584, 585, 586, 587, 588, 589, 591, 593, 594, 595, 596, 598, 600, 601, 602, 603, 604, 606, 607, 608, 609, 610, 612, 613, 614, 615, 616, 617, 620, 622, 623, 625, 628, 629, 631, 634, 637, 638, 640, 641, 643, 646, 647, 648, 649, 652, 654, 655, 656, 657, 658, 660, 662, 665, 666, 668, 669, 671, 674, 675, 676, 677, 678, 679, 680, 681, 682, 683, 684, 685, 686, 687, 688, 689, 691, 693, 694, 698, 699, 701, 702, 

In [16]:
# The complete list is also available as a Python variable for further analysis.
wrong_question_ids

[7,
 11,
 14,
 19,
 24,
 27,
 49,
 51,
 60,
 67,
 68,
 72,
 77,
 78,
 79,
 83,
 84,
 85,
 88,
 89,
 90,
 91,
 92,
 94,
 98,
 102,
 105,
 112,
 116,
 119,
 121,
 130,
 132,
 142,
 143,
 148,
 150,
 153,
 154,
 155,
 158,
 162,
 165,
 166,
 168,
 177,
 178,
 185,
 191,
 192,
 193,
 195,
 201,
 205,
 209,
 216,
 221,
 225,
 236,
 247,
 257,
 265,
 269,
 272,
 276,
 277,
 279,
 283,
 284,
 285,
 290,
 304,
 321,
 327,
 336,
 367,
 373,
 376,
 379,
 395,
 396,
 415,
 417,
 425,
 435,
 439,
 441,
 444,
 446,
 448,
 450,
 452,
 454,
 462,
 464,
 467,
 470,
 472,
 473,
 481,
 488,
 493,
 497,
 500,
 501,
 505,
 527,
 534,
 535,
 536,
 539,
 554,
 559,
 572,
 575,
 584,
 585,
 586,
 587,
 588,
 589,
 591,
 593,
 594,
 595,
 596,
 598,
 600,
 601,
 602,
 603,
 604,
 606,
 607,
 608,
 609,
 610,
 612,
 613,
 614,
 615,
 616,
 617,
 620,
 622,
 623,
 625,
 628,
 629,
 631,
 634,
 637,
 638,
 640,
 641,
 643,
 646,
 647,
 648,
 649,
 652,
 654,
 655,
 656,
 657,
 658,
 660,
 662,
 665,
 666,
 668,
